<a href="https://colab.research.google.com/github/NinhHTK/Documents/blob/main/Face_Regconition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **MỤC I: THU THẬP & TIỀN XỬ LÝ DỮ LIỆU**

In [19]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ashwingupta3012/human-faces")

print("Path to dataset files:", path)

100%|██████████| 1.82G/1.82G [00:26<00:00, 73.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ashwingupta3012/human-faces/versions/1


In [20]:
import os
import shutil
import glob

# Thiết lập đường dẫn đích
STRANGERS_DIR = 'face_dataset/strangers'
ME_DIR = 'face_dataset/me'
os.makedirs(STRANGERS_DIR, exist_ok=True)
os.makedirs(ME_DIR, exist_ok=True)

# Tìm tất cả các ảnh trong thư mục kagglehub vừa tải xuống
kaggle_images = glob.glob(os.path.join(path, '**/*.jpg'), recursive=True) + \
                glob.glob(os.path.join(path, '**/*.png'), recursive=True) + \
                glob.glob(os.path.join(path, '**/*.jpeg'), recursive=True)

print(f"Tìm thấy tổng cộng {len(kaggle_images)} ảnh người lạ trên Kaggle.")

# Chỉ lấy tối đa 50 - 100 ảnh để cân bằng với số lượng ảnh của bạn
NUM_STRANGERS = 50
selected_images = kaggle_images[:NUM_STRANGERS]

for idx, img_path in enumerate(selected_images):
    shutil.copy(img_path, os.path.join(STRANGERS_DIR, f"stranger_{idx}.jpg"))

print(f"Đã sao chép thành công {len(selected_images)} ảnh người lạ vào '{STRANGERS_DIR}' để cân bằng dữ liệu!")

Tìm thấy tổng cộng 7201 ảnh người lạ trên Kaggle.
Đã sao chép thành công 50 ảnh người lạ vào 'face_dataset/strangers' để cân bằng dữ liệu!


In [39]:
import os
import cv2
import face_recognition
import gdown
import shutil

# 1. Tải video trực tiếp từ Google Drive sử dụng File ID
file_id = '1Wk5rNYnub4NvVQhCtZU_5W23Eb9UWSdL'
local_video_path = 'my_uploaded_video.mp4'

print("Đang tải video từ Google Drive về Colab...")
url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(url, local_video_path, quiet=False)

# 2. Cấu hình thư mục đầu ra
OUTPUT_DIR = 'face_dataset/me'

# Khởi động lại thư mục để kiểm soát đúng số lượng 50 hình
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. Tiến hành đọc video và cắt đúng 50 khuôn mặt
target_count = 50
saved_count = 0
frame_count = 0

if not os.path.exists(local_video_path):
    print(f"[LỖI] Không thể tải video về máy ảo Colab.")
else:
    cap = cv2.VideoCapture(local_video_path)
    # Điều chỉnh khoảng cách khung hình nhỏ hơn để chắc chắn lấy đủ 50 ảnh từ video ngắn
    FRAME_INTERVAL = 5

    print(f"Bắt đầu trích xuất chính xác {target_count} khuôn mặt từ video...")
    while cap.isOpened() and saved_count < target_count:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % FRAME_INTERVAL == 0:
            # Đổi hệ màu sang RGB
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Phát hiện vị trí các khuôn mặt
            face_locations = face_recognition.face_locations(rgb_frame)

            for face_location in face_locations:
                if saved_count >= target_count:
                    break

                top, right, bottom, left = face_location
                # Trích xuất khuôn mặt
                face_image = frame[top:bottom, left:right]

                if face_image.size > 0:
                    saved_count += 1
                    output_path = os.path.join(OUTPUT_DIR, f"me_{saved_count}.jpg")
                    cv2.imwrite(output_path, face_image)

        frame_count += 1

    cap.release()
    # Xóa file video sau khi trích xuất xong để giải phóng dung lượng bộ nhớ tạm
    if os.path.exists(local_video_path):
        os.remove(local_video_path)

    print(f"\n[THÀNH CÔNG] Đã trích xuất hoàn tất!")
    print(f"Tổng số ảnh của bạn hiện tại: {len(os.listdir(OUTPUT_DIR))} / {target_count} ảnh.")

Đang tải video từ Google Drive về Colab...


Downloading...
From: https://drive.google.com/uc?id=1Wk5rNYnub4NvVQhCtZU_5W23Eb9UWSdL
To: /content/my_uploaded_video.mp4
100%|██████████| 31.0M/31.0M [00:01<00:00, 18.8MB/s]


Bắt đầu trích xuất chính xác 50 khuôn mặt từ video...

[THÀNH CÔNG] Đã trích xuất hoàn tất!
Tổng số ảnh của bạn hiện tại: 50 / 50 ảnh.


### **MỤC II: TRÍCH XUẤT ĐẶC TRƯNG KHUÔN MẶT (128-D)**

In [41]:
import os
import face_recognition
import numpy as np
import pickle
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# 1. Định nghĩa thư mục chứa dữ liệu
ME_DIR = 'face_dataset/me'
STRANGERS_DIR = 'face_dataset/strangers'

X = []
y = []

print("--- BẮT ĐẦU TRÍCH XUẤT FACE EMBEDDINGS (128-D) ---")

# Trích xuất embeddings cho ảnh của bạn (Nhãn: 1)
me_count = 0
for img_name in sorted(os.listdir(ME_DIR)):
    img_path = os.path.join(ME_DIR, img_name)
    try:
        image = face_recognition.load_image_file(img_path)
        encodings = face_recognition.face_encodings(image)
        if len(encodings) > 0:
            X.append(encodings[0])
            y.append(1)
            me_count += 1
    except Exception as e:
        print(f"Bỏ qua ảnh lỗi {img_name}: {e}")

print(f"Đã trích xuất thành công: {me_count} ảnh của bạn.")

# Trích xuất embeddings cho ảnh người lạ (Nhãn: 0)
stranger_count = 0
for img_name in sorted(os.listdir(STRANGERS_DIR)):
    img_path = os.path.join(STRANGERS_DIR, img_name)
    try:
        image = face_recognition.load_image_file(img_path)
        encodings = face_recognition.face_encodings(image)
        if len(encodings) > 0:
            X.append(encodings[0])
            y.append(0)
            stranger_count += 1
    except Exception as e:
        print(f"Bỏ qua ảnh lỗi {img_name}: {e}")

print(f"Đã trích xuất thành công: {stranger_count} ảnh người lạ.")

# Chuyển đổi sang mảng numpy
X = np.array(X)
y = np.array(y)

print(f"\nTổng mẫu dữ liệu hợp lệ đưa vào huấn luyện: {len(X)}")

if len(X) == 0:
    print("[LỖI] Không trích xuất được đặc trưng khuôn mặt nào. Vui lòng kiểm tra lại ảnh trong các thư mục.")
else:
    # 2. Phân chia tập dữ liệu huấn luyện và kiểm thử (80% Train, 20% Test)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # 3. Khởi tạo và huấn luyện mô hình phân loại SVM
    print("\n--- ĐANG HUẤN LUYỆN MÔ HÌNH SVM ---")
    classifier = SVC(kernel='linear', probability=True, random_state=42)
    classifier.fit(X_train, y_train)

    # 4. Đánh giá chất lượng mô hình
    y_pred = classifier.predict(X_test)
    print("\n--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---")
    print(f"Độ chính xác (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print("\nChi tiết báo cáo phân loại:")
    print(classification_report(y_test, y_pred, target_names=['Stranger', 'Me']))

    # 5. Lưu mô hình đã huấn luyện thành tệp tin pickle
    model_filename = 'classifier.pkl'
    with open(model_filename, 'wb') as f:
        pickle.dump(classifier, f)
    print(f"\n[THÀNH CÔNG] Đã lưu mô hình nhận diện khuôn mặt mới tại: '{model_filename}'!")

--- BẮT ĐẦU TRÍCH XUẤT FACE EMBEDDINGS (128-D) ---
Đã trích xuất thành công: 40 ảnh của bạn.
Đã trích xuất thành công: 46 ảnh người lạ.

Tổng mẫu dữ liệu hợp lệ đưa vào huấn luyện: 86

--- ĐANG HUẤN LUYỆN MÔ HÌNH SVM ---

--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---
Độ chính xác (Accuracy): 100.00%

Chi tiết báo cáo phân loại:
              precision    recall  f1-score   support

    Stranger       1.00      1.00      1.00        10
          Me       1.00      1.00      1.00         8

    accuracy                           1.00        18
   macro avg       1.00      1.00      1.00        18
weighted avg       1.00      1.00      1.00        18


[THÀNH CÔNG] Đã lưu mô hình nhận diện khuôn mặt mới tại: 'classifier.pkl'!


### **MỤC III: HUẤN LUYỆN MÔ HÌNH SVM**

In [42]:
import os
import face_recognition
import numpy as np
import pickle
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# 1. Định nghĩa thư mục chứa dữ liệu chuẩn
ME_DIR = 'face_dataset/me'
STRANGERS_DIR = 'face_dataset/strangers'

X = []
y = []

print("--- BẮT ĐẦU TRÍCH XUẤT FACE EMBEDDINGS (128-D) MỚI ---")

# Trích xuất embeddings cho 50 ảnh của bạn (Nhãn: 1)
me_count = 0
for img_name in sorted(os.listdir(ME_DIR)):
    img_path = os.path.join(ME_DIR, img_name)
    try:
        image = face_recognition.load_image_file(img_path)
        encodings = face_recognition.face_encodings(image)
        if len(encodings) > 0:
            X.append(encodings[0])
            y.append(1)
            me_count += 1
    except Exception as e:
        print(f"Bỏ qua ảnh lỗi {img_name}: {e}")

print(f"Đã trích xuất thành công: {me_count} ảnh của bạn.")

# Trích xuất embeddings cho 50 ảnh người lạ (Nhãn: 0)
stranger_count = 0
for img_name in sorted(os.listdir(STRANGERS_DIR)):
    img_path = os.path.join(STRANGERS_DIR, img_name)
    try:
        image = face_recognition.load_image_file(img_path)
        encodings = face_recognition.face_encodings(image)
        if len(encodings) > 0:
            X.append(encodings[0])
            y.append(0)
            stranger_count += 1
    except Exception as e:
        print(f"Bỏ qua ảnh lỗi {img_name}: {e}")

print(f"Đã trích xuất thành công: {stranger_count} ảnh người lạ.")

# Chuyển đổi sang mảng numpy
X = np.array(X)
y = np.array(y)

print(f"\nTổng mẫu dữ liệu hợp lệ đưa vào huấn luyện: {len(X)}")

# 2. Phân chia tập dữ liệu huấn luyện và kiểm thử (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Khởi tạo và huấn luyện mô hình phân loại SVM
print("\n--- ĐANG HUẤN LUYỆN MÔ HÌNH SVM MỚI ---")
classifier = SVC(kernel='linear', probability=True, random_state=42)
classifier.fit(X_train, y_train)

# 4. Đánh giá chất lượng mô hình
y_pred = classifier.predict(X_test)
print("\n--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---")
print(f"Độ chính xác (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nChi tiết báo cáo phân loại:")
print(classification_report(y_test, y_pred, target_names=['Stranger', 'Me']))

# 5. Lưu mô hình đã huấn luyện thành tệp tin pickle
model_filename = 'classifier.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(classifier, f)
print(f"\n[THÀNH CÔNG] Đã lưu mô hình nhận diện khuôn mặt tối ưu tại: '{model_filename}'!")

--- BẮT ĐẦU TRÍCH XUẤT FACE EMBEDDINGS (128-D) MỚI ---
Đã trích xuất thành công: 40 ảnh của bạn.
Đã trích xuất thành công: 46 ảnh người lạ.

Tổng mẫu dữ liệu hợp lệ đưa vào huấn luyện: 86

--- ĐANG HUẤN LUYỆN MÔ HÌNH SVM MỚI ---

--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---
Độ chính xác (Accuracy): 100.00%

Chi tiết báo cáo phân loại:
              precision    recall  f1-score   support

    Stranger       1.00      1.00      1.00        10
          Me       1.00      1.00      1.00         8

    accuracy                           1.00        18
   macro avg       1.00      1.00      1.00        18
weighted avg       1.00      1.00      1.00        18


[THÀNH CÔNG] Đã lưu mô hình nhận diện khuôn mặt tối ưu tại: 'classifier.pkl'!


### **MỤC IV: NHẬN DIỆN THỜI GIAN THỰC (REAL-TIME)**

In [43]:
inference_script_content = """
import cv2
import pickle
import numpy as np
import face_recognition

# 1. Tải mô hình đã được huấn luyện
try:
    with open('classifier.pkl', 'rb') as f:
        clf = pickle.load(f)
    print("[INFO] Đã tải mô hình classifier.pkl thành công.")
except Exception as e:
    print(f"[ERROR] Không thể tải mô hình: {e}. Vui lòng đảm bảo file 'classifier.pkl' nằm cùng thư mục với file script này.")
    exit()

# Ngưỡng an toàn độ tin cậy nhận diện (ví dụ: >= 75% mới nhận diện là bạn)
THRESHOLD = 0.75

# 2. Khởi tạo kết nối tới camera của máy tính cá nhân
video_capture = cv2.VideoCapture(0)

print("Đang khởi động Webcam... Nhấn phím 'q' để thoát khỏi chương trình.")

while True:
    ret, frame = video_capture.read()
    if not ret:
        print("[LỖI] Không thể đọc luồng dữ liệu từ camera.")
        break

    # Resize khung hình xuống tỉ lệ 1/4 để tăng tốc độ nhận diện thời gian thực
    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    # Phát hiện tất cả khuôn mặt và trích xuất vector đặc trưng
    face_locations = face_recognition.face_locations(rgb_small_frame)
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        # Dự đoán phân loại xác suất
        probs = clf.predict_proba([face_encoding])[0]
        class_idx = np.argmax(probs)
        confidence = probs[class_idx]

        # Gán nhãn dựa trên kết quả phân loại và ngưỡng tin cậy
        if class_idx == 1 and confidence >= THRESHOLD:
            label = f"ME ({confidence*100:.1f}%)"
            box_color = (0, 255, 0) # Màu xanh lá
        else:
            label = f"STRANGER ({probs[0]*100:.1f}%)"
            box_color = (0, 0, 255) # Màu đỏ

        # Khôi phục tỷ lệ khung hình ban đầu (x4)
        top *= 4
        right *= 4
        bottom *= 4
        left *= 4

        # Vẽ khung bao quanh khuôn mặt
        cv2.rectangle(frame, (left, top), (right, bottom), box_color, 2)

        # Vẽ nhãn tên
        cv2.rectangle(frame, (left, bottom - 35), (right, bottom), box_color, cv2.FILLED)
        font = cv2.FONT_HERSHEY_DUPLEX
        cv2.putText(frame, label, (left + 6, bottom - 6), font, 0.6, (255, 255, 255), 1)

    # Hiển thị kết quả lên màn hình
    cv2.imshow('Real-time Face Recognition System', frame)

    # Bấm 'q' để kết thúc chương trình
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Giải phóng camera
video_capture.release()
cv2.destroyAllWindows()
print("Chương trình kết thúc.")
"""

with open('realtime_inference.py', 'w', encoding='utf-8') as f:
    f.write(inference_script_content)

print("[THÀNH CÔNG] Đã tạo file 'realtime_inference.py'. Hãy tải xuống máy tính cá nhân để thực thi.")

[THÀNH CÔNG] Đã tạo file 'realtime_inference.py'. Hãy tải xuống máy tính cá nhân để thực thi.


### Trình kiểm thử trực tuyến (Online Inference Test)
Sử dụng ô mã nguồn dưới đây để tải lên một hình ảnh bất kỳ từ máy tính của bạn lên Google Colab và xem mô hình nhận diện khuôn mặt trực tuyến.

In [45]:
import cv2
import pickle
import numpy as np
import face_recognition
from google.colab import files
import matplotlib.pyplot as plt

# 1. Tải mô hình đã được huấn luyện
try:
    with open('classifier.pkl', 'rb') as f:
        clf = pickle.load(f)
    print("[INFO] Đã tải mô hình classifier.pkl thành công.\n")
except Exception as e:
    print(f"[ERROR] Không thể tải mô hình: {e}. Vui lòng chạy lại bước huấn luyện mô hình trước.")
    clf = None

if clf is not None:
    # 2. Upload ảnh từ máy tính cá nhân lên Colab để test
    print("Hãy chọn một ảnh từ máy tính của bạn (định dạng .jpg, .png, .jpeg) để tải lên...")
    uploaded = files.upload()

    for filename in uploaded.keys():
        # Kiểm tra định dạng file tải lên để tránh lỗi UnidentifiedImageError
        ext = filename.split('.')[-1].lower()
        if ext not in ['jpg', 'jpeg', 'png', 'bmp', 'webp']:
            print(f"\n[CẢNH BÁO] Tệp \"{filename}\" không phải là định dạng ảnh hợp lệ (jpg, jpeg, png, bmp, webp). Vui lòng tải lại ảnh thực tế.")
            continue

        print(f'\nĐang xử lý ảnh: "{filename}"')

        try:
            # Đọc ảnh vừa tải lên
            image = face_recognition.load_image_file(filename)

            # Phát hiện khuôn mặt và trích xuất vector đặc trưng
            face_locations = face_recognition.face_locations(image)
            face_encodings = face_recognition.face_encodings(image, face_locations)

            # Đọc ảnh bằng OpenCV để vẽ hình và hiển thị
            bgr_image = cv2.imread(filename)
            rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)

            if len(face_encodings) == 0:
                print("[CẢNH BÁO] Không tìm thấy khuôn mặt nào trong bức ảnh này!")
            else:
                print(f"Tìm thấy {len(face_encodings)} khuôn mặt trong bức ảnh.")

                # Ngưỡng tin cậy
                THRESHOLD = 0.75

                for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
                    # Dự đoán xác suất bằng mô hình SVM
                    probs = clf.predict_proba([face_encoding])[0]
                    class_idx = np.argmax(probs)
                    confidence = probs[class_idx]

                    # Xác định nhãn phân loại
                    if class_idx == 1 and confidence >= THRESHOLD:
                        label = f"ME ({confidence*100:.1f}%)"
                        box_color = (0, 255, 0)  # Xanh lá cho bạn
                    else:
                        label = f"STRANGER ({probs[0]*100:.1f}%)"
                        box_color = (255, 0, 0)  # Đỏ cho người lạ

                    # Vẽ hình vuông bao quanh khuôn mặt
                    cv2.rectangle(rgb_image, (left, top), (right, bottom), box_color, 4)

                    # Hiển thị nhãn trên ảnh
                    font = cv2.FONT_HERSHEY_DUPLEX
                    cv2.putText(rgb_image, label, (left, top - 15), font, 1.0, box_color, 2)
                    print(f"Kết quả: {label}")

                # Hiển thị ảnh kết quả cuối cùng bằng Matplotlib
                plt.figure(figsize=(10, 8))
                plt.imshow(rgb_image)
                plt.axis('off')
                plt.show()
        except Exception as img_err:
            print(f"[LỖI] Đã xảy ra lỗi khi xử lý ảnh này: {img_err}")

[INFO] Đã tải mô hình classifier.pkl thành công.

Hãy chọn một ảnh từ máy tính của bạn (định dạng .jpg, .png, .jpeg) để tải lên...


Saving classifier.pkl to classifier (2).pkl

[CẢNH BÁO] Tệp "classifier (2).pkl" không phải là định dạng ảnh hợp lệ (jpg, jpeg, png, bmp, webp). Vui lòng tải lại ảnh thực tế.


In [49]:
import cv2
import pickle
import numpy as np
import face_recognition
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import io
from PIL import Image

try:
    with open('classifier.pkl', 'rb') as f:
        clf = pickle.load(f)
    print("[INFO] Đã tải mô hình classifier.pkl.")
except Exception as e:
    print(f"[ERROR] Lỗi tải mô hình: {e}")
    clf = None

def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    return cv2.imdecode(jpg_as_np, flags=1)

def bbox_to_bytes(bbox_array):
    bbox_PIL = Image.fromarray(bbox_array.astype(np.uint8))
    io_buf = io.BytesIO()
    bbox_PIL.save(io_buf, format='png')
    return 'data:image/png;base64,{}'.format((str(b64encode(io_buf.getvalue()), 'utf-8')))

def start_webcam():
    js = Javascript('''
    var video, div = null, stream, captureCanvas, imgElement;
    var pendingResolve = null, shutdown = false;

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = div = stream = imgElement = captureCanvas = null;
    }

    function onAnimationFrame() {
      if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext(\'2d\').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL(\'image/jpeg\', 0.8);
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) return;
      div = document.createElement(\'div\');
      div.style.border = \'2px solid black\';
      div.style.padding = \'4px\';
      div.style.width = \'640px\';
      div.style.margin = \'auto\';

      const highlight = document.createElement(\'div\');
      highlight.innerHTML = "<h5>Hệ thống Nhận diện Khuôn mặt</h5>";
      highlight.style.textAlign = \'center\';
      div.appendChild(highlight);

      video = document.createElement(\'video\');
      video.style.display = \'block\';
      video.width = 640;
      video.height = 480;
      video.autoplay = video.playsInline = true;
      div.appendChild(video);

      const btn = document.createElement(\'button\');
      btn.textContent = \'Dừng nhận diện\';
      btn.style.cssText = \'display:block; width:100%; margin-top:10px; padding:8px; background-color:#ff4d4d; color:white; border:none; cursor:pointer;\';
      btn.onclick = () => { shutdown = true; };
      div.appendChild(btn);

      document.body.appendChild(div);

      stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: \'user\'}});
      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement(\'canvas\');
      captureCanvas.width = 640;
      captureCanvas.height = 480;

      imgElement = document.createElement(\'img\');
      imgElement.style.cssText = \'position:absolute; pointer-events:none;\';
      imgElement.style.top = video.offsetTop + \'px\';
      imgElement.style.left = video.offsetLeft + \'px\';
      imgElement.style.width = video.offsetWidth + \'px\';
      imgElement.style.height = video.offsetHeight + \'px\';
      div.appendChild(imgElement);

      window.requestAnimationFrame(onAnimationFrame);
    }

    async function streamFrame(rects) {
      if (div === null) await createDom();
      if (shutdown) {
        removeDom();
        return "shutdown";
      }
      imgElement.src = rects || "data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP///yH5BAEAAAAALAAAAAABAAEAAAIBRAA7";
      return new Promise((resolve) => { pendingResolve = resolve; });
    }
    ''')
    display(js)

if clf is not None:
    start_webcam()
    js_reply = eval_js('streamFrame("")')
    THRESHOLD = 0.75

    while True:
        if js_reply == 'shutdown' or not js_reply:
            print("\n[INFO] Đã đóng camera.")
            break

        img = js_to_image(js_reply)
        overlay = np.zeros((480, 640, 4), dtype=np.uint8)

        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        face_locations = face_recognition.face_locations(rgb_img)
        face_encodings = face_recognition.face_encodings(rgb_img, face_locations)

        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            probs = clf.predict_proba([face_encoding])[0]
            class_idx = np.argmax(probs)
            confidence = probs[class_idx]

            if class_idx == 1 and confidence >= THRESHOLD:
                label = f"ME ({confidence*100:.1f}%)"
                color = (0, 255, 0, 255)
            else:
                label = f"STRANGER ({probs[0]*100:.1f}%)"
                color = (255, 0, 0, 255)

            cv2.rectangle(overlay, (left, top), (right, bottom), color, 3)
            cv2.putText(overlay, label, (left, top - 10), cv2.FONT_HERSHEY_DUPLEX, 0.6, color, 1)

        overlay_bytes = bbox_to_bytes(overlay)
        js_reply = eval_js(f'streamFrame("{overlay_bytes}")')

[INFO] Đã tải mô hình classifier.pkl.


<IPython.core.display.Javascript object>


[INFO] Đã đóng camera.
